# Notebook 03: Semi-Supervised Learning

**Module:** 04 ML Paradigms  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Understand learning with few labels + many unlabeled samples
2. Implement pseudo-labeling
3. Apply Label Propagation
4. Know when semi-supervised beats supervised

## 1. Intuition

**Problem:** Labeling is expensive. You have 100 labeled images and 10,000 unlabeled.

**Semi-supervised learning** uses both labeled and unlabeled data.

**Assumption (smoothness):** Points close in feature space should have similar labels.

## 2. Methods

| Method | Idea |
|--------|------|
| **Pseudo-labeling** | Train on labeled, predict unlabeled, add confident predictions to training set, repeat |
| **Label Propagation** | Spread labels through a similarity graph |
| **Self-training** | Model generates its own training data |
| **MixMatch / FixMatch** | Deep semi-supervised (Module 05+) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

REPO = Path('../../').resolve()
plt.rcParams['figure.figsize'] = (8, 5)
rng = np.random.default_rng(42)

In [ ]:
from sklearn.semi_supervised import LabelPropagation
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=500, n_features=10, random_state=42)

# Hide 90% of labels (set to -1)
y_semi = y.copy()
mask = rng.random(len(y)) < 0.9
y_semi[mask] = -1
print(f'Labeled: {(y_semi >= 0).sum()}, Unlabeled: {(y_semi < 0).sum()}')

lp = LabelPropagation(kernel='knn', n_neighbors=7).fit(X, y_semi)
print(f'Label Propagation accuracy: {(lp.predict(X) == y).mean():.4f}')

In [ ]:
# Pseudo-labeling demo
def pseudo_label(X_l, y_l, X_u, base_clf, threshold=0.9):
    base_clf.fit(X_l, y_l)
    proba = base_clf.predict_proba(X_u)
    conf = proba.max(axis=1)
    preds = proba.argmax(axis=1)
    confident = conf >= threshold
    X_new = np.vstack([X_l, X_u[confident]])
    y_new = np.concatenate([y_l, preds[confident]])
    return X_new, y_new, confident.sum()

from sklearn.linear_model import LogisticRegression
X_l, X_u, y_l, y_u = X[~mask], X[mask], y[~mask], y[mask]
clf = LogisticRegression(max_iter=1000)
X_new, y_new, n_added = pseudo_label(X_l, y_l, X_u, clf)
print(f'Added {n_added} pseudo-labeled samples')

## GeoSpatial Example

10 labeled aquaculture ponds + 5000 unlabeled tiles → semi-supervised segmentation reduces labeling cost by 90%.

## Exercise

Implement 3 iterations of pseudo-labeling. Plot accuracy vs iteration.

In [ ]:
# YOUR CODE HERE


---

## Interview Questions

See module quiz.

## Summary

Semi-supervised learning leverages cheap unlabeled data to boost limited labels.

**Next:** [04_Self_Supervised_Learning.ipynb](04_Self_Supervised_Learning.ipynb)